# 01b · PhysioNet 2019 cross-cohort generalisation (Track A2)

Do the models generalise to another hospital? External validation in three directions:

1. **MIMIC-IV FIT → PhysioNet set A and set B** (primary).
2. **PhysioNet (A+B) → MIMIC-IV VAL** (the KA paper's direction).
3. **set A ↔ set B** (a cheap site-shift check).

**Which set is truly external.** Set B (Emory) is the clean external test. **Set A is Beth Israel Deaconess, the hospital behind MIMIC**: it is a temporal / same-site check with possible patient overlap, reported separately.

**Rules.**
- Only the **33 features both cohorts have** are used; `GCS`, `UrineOut_mL`, `PF_ratio` (MIMIC only) and `TroponinI` (PhysioNet only) are dropped. All cross-cohort models are therefore **reduced-feature models retrained on MIMIC FIT**; their MIMIC VAL score is shown next to the full-feature model so the cost of dropping features is visible.
- External data is **never** used to fit normalisation, calibration or thresholds. MIMIC FIT statistics are applied as they are. Raw and MIMIC-calibrated probabilities are reported; recalibration on a small external split is a labelled secondary result.
- Same label definition in both cohorts. **Primary comparison = matched stratum** (stay ≥ 8 h; non-septic, or onset ≥ 4 h and onset not censored), because the Challenge excluded onset < 4 h and records < 8 h, while MIMIC contains many such stays. All stays are reported beside it.
- The two cohorts' Sepsis-3 labels were derived by different pipelines, so an external drop mixes label shift with covariate shift.

**This notebook stops after the harmonisation table** until you have read it and set `HARMONISATION_REVIEWED = True` (units must be verified before anything trains).

The attention replication on set B (P1–P4 with a partial-SOFA proxy) is **not** here: it needs the correlational battery, which is built at the start of Phase 3.

In [ ]:
# ============================ CELL 1 · BOOTSTRAP: clone the repo, import sepsis_core from it ============================
REPO_URL = "https://github.com/Myakey/sepsis-transformer-analysis.git"
REPO_REF = "main"                                  # branch, tag or commit hash. PIN A COMMIT HASH for any run you will report.
REPO_DIR = "/content/sepsis-transformer-analysis"

import os, sys, subprocess
sys.dont_write_bytecode = True
if "google.colab" in sys.modules and not os.path.exists("/content/drive/MyDrive"):      # Drive holds the data and the checkpoints
    from google.colab import drive
    drive.mount("/content/drive")

def _git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(f"git {' '.join(args[:2])} failed: {r.stderr.strip()[-300:]}")
    return r.stdout.strip()

def _auth_url(url):
    tok = None                                     # private repo: Colab secret GITHUB_TOKEN (Notebook access on)
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        tok = os.environ.get("GITHUB_TOKEN")
    return url.replace("https://", f"https://{tok}@") if tok else url

if not os.path.isdir(os.path.join(REPO_DIR, ".git")): _git("clone", "--quiet", _auth_url(REPO_URL), REPO_DIR)
else: _git("fetch", "--quiet", "--all", cwd=REPO_DIR)
_git("checkout", "--quiet", REPO_REF, cwd=REPO_DIR)
if REPO_REF in _git("branch", "--format=%(refname:short)", cwd=REPO_DIR).split(): _git("pull", "--quiet", "--ff-only", cwd=REPO_DIR)
_git("remote", "set-url", "origin", REPO_URL, cwd=REPO_DIR)        # never leave a token in the clone's config
CORE_COMMIT = _git("rev-parse", "HEAD", cwd=REPO_DIR)
os.environ["SEPSIS_PROJECT_ROOT"] = REPO_DIR
os.environ["SEPSIS_CORE_COMMIT"] = CORE_COMMIT
for m in [k for k in sys.modules if k == "sepsis_core" or k.startswith("sepsis_core.")]: del sys.modules[m]
if os.path.join(REPO_DIR, "Notebooks", "v4") not in sys.path: sys.path.insert(0, os.path.join(REPO_DIR, "Notebooks", "v4"))

import importlib.util
_need = [p for p in ("mlflow", "lightgbm") if importlib.util.find_spec(p) is None]
if _need: subprocess.call([sys.executable, "-m", "pip", "install", "-q", *_need])

import json, time, gc
import numpy as np, pandas as pd, torch
import sepsis_core
from sepsis_core import config as C, data, dataset, models, engine, metrics, calibration, balancing, controls, gbdt, runs, plots, tracking, physionet
print("sepsis_core", sepsis_core.__version__, "@ commit", CORE_COMMIT[:12], f"({REPO_REF})", "| torch", torch.__version__)

In [ ]:
# ============================ CELL 2 · SWITCHES ============================
SMOKE_TEST = False
HARMONISATION_REVIEWED = False     # set True only after reading the table printed by Cell 5
UNIT_FIXES = {}                    # {PhysioNet column: multiplier}, e.g. {"FiO2": 100.0} if PhysioNet stores a fraction and MIMIC a percent
DIRECTIONS = dict(mimic_to_physionet=True, physionet_to_mimic=True, setA_vs_setB=True)
ICULOS_OPTS = (False, True)        # direction 1 only; directions 2 and 3 run without the explicit ICULOS input
ARMS_D1 = ("softmax", "sigmoid", "ka")

runs.setup(smoke=SMOKE_TEST)
SEED = C.CFG.SEEDS[0]
HP = dict(C.V3_BEST_HP)
FEATS = physionet.reduced_features()
PHYS_LIMIT = 400 if SMOKE_TEST else None
REC, RESULTS = {}, {}
def hp_arm(arm, ic=False): return {**HP, "arm": arm, "use_iculos": ic, **({"aux_w": 0.0} if arm == "ka" else {})}
print(len(FEATS), "shared features; dropped from MIMIC:", physionet.MIMIC_ONLY, "| dropped from PhysioNet:", physionet.PHYSIONET_ONLY)

In [ ]:
# ============================ CELL 3 · MIMIC-IV, reduced features (FIT/VAL) ============================
R = runs.prepare_mimic(features=FEATS, tag="r33", keep_df=True)

In [ ]:
# ============================ CELL 4 · PhysioNet sets A and B (raw) ============================
dirs = physionet.extract()
RAW = {s: physionet.read_set(dirs[s], s, limit=PHYS_LIMIT) for s in ("A", "B")}
for s in RAW: print(f"set {s}: {RAW[s][C.CFG.ID_COL].nunique():,} patients, {len(RAW[s]):,} hourly rows, raw positive-row rate {RAW[s]['SepsisLabel'].mean():.4f}")

## Harmonisation check (read this before going on)

Per feature and cohort: the share of hours with a measurement, the median, and the 5th–95th percentile. A `CHECK UNITS` flag means a cohort's median is outside 0.67×–1.5× of MIMIC FIT; `CHECK DENSITY` means the measured share differs by more than 0.20. For each `CHECK UNITS`, decide whether it is a unit difference (then add a multiplier to `UNIT_FIXES` in Cell 2) or a real population difference (then leave it). Copy the saved table into `Docs/HARMONISATION.md`.

In [ ]:
# ============================ CELL 5 · HARMONISATION TABLE (aggregates) ============================
frames = {"MIMIC_FIT": R["df"][R["df"]["fold"] == "FIT"], "setA": RAW["A"], "setB": RAW["B"]}
HARM = physionet.harmonisation_table(frames, FEATS)
print(f"{'feature':<18s}{'measured share  M / A / B':>30s}{'median  M / A / B':>36s}   flag")
for r in HARM:
    print(f"{r['feature']:<18s}{r['MIMIC_FIT_measured']:>10.3f}{r['setA_measured']:>10.3f}{r['setB_measured']:>10.3f}"
          f"{r['MIMIC_FIT_median']:>12.3f}{r['setA_median']:>12.3f}{r['setB_median']:>12.3f}   {r['flag']}")
runs.save_json(dict(unit_fixes_applied=UNIT_FIXES, table=HARM, dropped_mimic_only=physionet.MIMIC_ONLY, dropped_physionet_only=physionet.PHYSIONET_ONLY), "01b_harmonisation.json")
n_flag = sum(1 for r in HARM if "UNITS" in r["flag"])
print(f"\n{n_flag} feature(s) flagged CHECK UNITS. UNIT_FIXES currently: {UNIT_FIXES}")
if not HARMONISATION_REVIEWED:
    raise RuntimeError("STOP: read the table above. Add multipliers to UNIT_FIXES for genuine unit differences, then set HARMONISATION_REVIEWED = True in Cell 2 and re-run from Cell 2.")

In [ ]:
# ============================ CELL 6 · PhysioNet in the MIMIC layout, with the project's label ============================
PH, ONSET, CENS, PSTAT = {}, {}, {}, {}
for s in ("A", "B"):
    PH[s], STATIC_P, PSTAT[s], meta = physionet.to_layout(RAW[s], s, features=FEATS, unit_fixes=UNIT_FIXES)
    ONSET[s] = data.stay_onset(PH[s]); CENS[s] = meta.index[meta["censored"]].to_numpy()
    st_ = PSTAT[s]
    print(f"set {s}: rows {st_['rows']:,} (raw {st_['rows_raw']:,}) | stays {st_['stays']:,} | positive rows {st_['pos_rate']:.4%} | septic stays {st_['septic_stays']:,} | "
          f"early-onset (<{C.CFG.EARLY_ONSET_HOURS} h) {st_['early_onset_share']:.1%} | onset censored (positive on first row) {st_['censored_share_of_septic']:.1%} of septic | "
          f"rebuilt label == Challenge label before onset: {st_['label_agreement_with_challenge_before_onset']:.4f} | first hour is 1 in {st_['first_hour_is_1']:.1%} | length {st_['min_len']}-{st_['max_len']} h")
DF_MIMIC_VAL = R["df"][R["df"]["fold"] == "VAL"].drop(columns=["fold"]).copy()
del RAW, frames; R.pop("df"); gc.collect()
M_MASK, m_info = physionet.matched_mask(R["store"], R["onset"])
o = R["onset"].dropna()
print(f"MIMIC (FIT+VAL): matched stratum keeps {m_info['stays_matched']:,}/{m_info['stays_total']:,} stays ({m_info['septic_matched']:,} septic) | "
      f"septic stays with onset < 4 h: {(o < 4).mean():.1%} | onset < {C.CFG.EARLY_ONSET_HOURS} h: {(o < C.CFG.EARLY_ONSET_HOURS).mean():.1%}")

## Direction 1 · MIMIC-IV FIT → PhysioNet A and B (primary)

In [ ]:
# ============================ CELL 7 · TRAIN the reduced-feature models on MIMIC FIT ============================
GBR = {}
if DIRECTIONS["mimic_to_physionet"]:
    for ic in ICULOS_OPTS:
        nm = f"r33_gbdt{'_iculos' if ic else ''}"; GBR[nm] = gbdt.fit_or_load(runs.model_dir(nm), R["store"], R["pre"], R["fit_idx"], R["val_idx"], with_iculos=ic)
        for arm in ARMS_D1:
            nm = f"r33_{arm}{'_iculos' if ic else ''}_s{SEED}"; REC[nm] = runs.train_model(R, nm, hp_arm(arm, ic), seed=SEED, role="crosscohort_mimic_reduced")
    print("\nCost of dropping GCS / urine output / PF ratio (MIMIC VAL AUROC, full-feature model from notebook 02 or 01 if it exists):")
    for arm in ARMS_D1:
        full = os.path.join(runs.model_dir(f"{arm}_s{SEED}"), "done.json"); red = REC.get(f"r33_{arm}_s{SEED}")
        f_ = json.load(open(full))["metrics"]["auroc_mean"] if os.path.exists(full) else None
        if red: print(f"  {arm:<8s} reduced (33) {red['metrics']['auroc_mean']:.4f} | full (36) {('%.4f' % f_) if f_ is not None else 'not trained yet'}" + (f" | cost {red['metrics']['auroc_mean'] - f_:+.4f}" if f_ is not None else ""))

In [ ]:
# ============================ CELL 8 · EVALUATE on set A and set B (MIMIC statistics applied as they are) ============================
def external_side(src, key, df_t, onset_t, censored_t=None):
    """Transform a target cohort with the SOURCE preprocessor; return its store, row-level onset and matched-stratum mask."""
    st = src["pre"].transform(df_t); orow = metrics.row_onset(st, onset_t); mask, info = physionet.matched_mask(st, onset_t, censored_t)
    print(f"  target {key}: {len(st.starts):,} stays, {st.n_rows:,} rows, hour-gap violations {st.gap_violations:,} | matched stratum {info['stays_matched']:,} stays ({info['septic_matched']:,} septic)")
    return st, orow, mask

def show(tag, name, ext, internal):
    for view in ("matched", "all"):
        if view not in ext: continue
        e = ext[view]; i = internal[view]
        print(f"  [{tag}] {name:<26s} {view:<8s} external AUROC {e['metrics']['auroc_mean']:.4f} [{e['auroc_mean_ci'][1]:.4f}, {e['auroc_mean_ci'][2]:.4f}] "
              f"(t1/t3/t6 {e['metrics']['auroc_t1']:.3f}/{e['metrics']['auroc_t3']:.3f}/{e['metrics']['auroc_t6']:.3f}) | internal {i['metrics']['auroc_mean']:.4f} | "
              f"drop {e['metrics']['auroc_mean'] - i['metrics']['auroc_mean']:+.4f} | AUPRC lift x{np.mean([e['metrics'][f'auprc_lift_t{h}'] for h in C.HORIZONS]):.1f}")
    c_raw, c_cal = ext["calibration_raw"], ext["all"].get("calibration", [])
    if c_raw and c_cal:
        print(f"       calibration on all stays: raw mean pred {np.mean([c['mean_pred'] for c in c_raw]):.4f} vs observed {np.mean([c['obs_rate'] for c in c_raw]):.4f} | "
              f"source-calibrated ({ext['source_calibrator']}): in-the-large {np.mean([c['cal_in_the_large'] for c in c_cal]):+.2f}, slope {np.mean([c['cal_slope'] for c in c_cal]):.2f}, ECE {np.mean([c['ece'] for c in c_cal]):.4f}")

def run_direction(tag, src, names, gbs, targets, src_mask):
    """names: neural models trained on `src`; gbs: {name: LightGBM dict}; targets: {key: (store, onset_rows, mask)}."""
    out = {}
    for nm in list(names) + list(gbs):
        gb = gbs.get(nm)
        internal = dict(matched=runs.internal_report(src, nm, src_mask, gb), all=runs.internal_report(src, nm, None, gb))
        out[nm] = dict(internal=internal)
        for key, (st, orow, mask) in targets.items():
            out[nm][key] = runs.external_report(src, nm, st, orow, mask, gb); show(f"{tag} -> {key}", nm, out[nm][key], internal)
    return out

TGT_R = {}
if DIRECTIONS["mimic_to_physionet"]:
    for s in ("A", "B"): TGT_R[f"set{s}"] = external_side(R, f"set{s}", PH[s], ONSET[s], CENS[s])
    RESULTS["mimic_to_physionet"] = run_direction("MIMIC", R, list(REC), GBR, TGT_R, M_MASK)
    print("\nReminder: setA is Beth Israel (same hospital as MIMIC; possible patient overlap). setB (Emory) is the clean external test.")

In [ ]:
# ============================ CELL 9 · BY-HOUR CURVES on the external sets + SHORTCUT DIAGNOSTICS ============================
if DIRECTIONS["mimic_to_physionet"]:
    nm = f"r33_softmax_s{SEED}"
    if nm in RESULTS["mimic_to_physionet"]:
        tabs = {f"softmax on {k} (matched)": RESULTS["mimic_to_physionet"][nm][k]["matched"]["by_icu_hour"] for k in TGT_R}
        plots.by_icu_hour(tabs, horizon=6, metric="auroc", path=runs.figure_path("01b_auroc_by_hour_external_t6.png"), title="MIMIC-trained Transformer on PhysioNet: AUROC by ICU hour (t+6 h)")
    print("Does a measurement-density shortcut exist, and would it transfer? (AUROC of density alone / of the ICU hour alone for the t+6 label)")
    DIAG = {"MIMIC_VAL": physionet.shortcut_diagnostics(R["store"], R["val_idx"])}
    for k, (st, _, _) in TGT_R.items(): DIAG[k] = physionet.shortcut_diagnostics(st)
    for k, d in DIAG.items():
        print(f"  {k:<10s} measured share/hour {d['measured_share_per_hour']:.3f} | features measured per hour: label 0 {d['features_measured_per_hour_label0']:.2f}, label 1 {d['features_measured_per_hour_label1']:.2f} | "
              f"AUROC density-only {d['auroc_density_only']:.3f} | 6 h density {d['auroc_density6h_only']:.3f} | ICU-hour-only {d['auroc_icu_hour_only']:.3f}")
    RESULTS["shortcut_diagnostics"] = {k: {kk: vv for kk, vv in d.items() if kk != "per_feature_measured_share"} for k, d in DIAG.items()}
    RESULTS["measured_share_per_feature"] = {k: dict(zip(FEATS, d["per_feature_measured_share"].values())) for k, d in DIAG.items()}

## Direction 2 · PhysioNet (A+B) → MIMIC-IV VAL

The KA paper's direction. Models are trained on PhysioNet with a patient-level FIT/VAL split inside PhysioNet (normalisation, early stopping and calibration all come from PhysioNet), then scored once on MIMIC VAL.

In [ ]:
# ============================ CELL 10 · PhysioNet -> MIMIC VAL ============================
def phys_bundle(sets, tag):
    dfp = physionet.split_patients(pd.concat([PH[s] for s in sets], ignore_index=True), 0.15, seed=0)
    prep = data.Preprocessor(features=FEATS, static_cols=STATIC_P).fit(dfp, (dfp["fold"] == "FIT").to_numpy(), fold=dfp["fold"].to_numpy())
    stats = dict(rows=int(len(dfp)), stays=int(dfp[C.CFG.ID_COL].nunique()), pos_rate=float(dfp["SepsisLabel"].mean()), sets="".join(sets))
    onset = pd.concat([ONSET[s] for s in sets]); cens = np.concatenate([CENS[s] for s in sets])
    b = runs.make_bundle(dfp, prep, STATIC_P, stats, onset, tag, split_sha="physionet-internal-split-seed0"); b["censored"] = cens
    b["mask"], _ = physionet.matched_mask(b["store"], onset, cens); return b

def train_on(b, prefix, arms):
    names, gbs = [], {}
    gbs[f"{prefix}_gbdt"] = gbdt.fit_or_load(runs.model_dir(f"{prefix}_gbdt"), b["store"], b["pre"], b["fit_idx"], b["val_idx"])
    for arm in arms:
        nm = f"{prefix}_{arm}_s{SEED}"; REC[nm] = runs.train_model(b, nm, hp_arm(arm), seed=SEED, role=f"crosscohort_{prefix}"); names.append(nm)
    return names, gbs

if DIRECTIONS["physionet_to_mimic"]:
    P = phys_bundle(("A", "B"), "physAB")
    names, gbs = train_on(P, "p2m", ("softmax", "ka"))
    tgt = {"MIMIC_VAL": external_side(P, "MIMIC_VAL", DF_MIMIC_VAL, R["onset"])}
    RESULTS["physionet_to_mimic"] = run_direction("PhysioNet", P, names, gbs, tgt, P["mask"])
    print("\nContext only (different task, not comparable): the KA paper reports AUROC 0.92-0.95 on MIMIC-IV for one prediction per patient.")
    del P; gc.collect()

## Direction 3 · set A ↔ set B (site shift inside PhysioNet)

In [ ]:
# ============================ CELL 11 · A -> B and B -> A ============================
if DIRECTIONS["setA_vs_setB"]:
    for src_s, tgt_s in (("A", "B"), ("B", "A")):
        Bs = phys_bundle((src_s,), f"phys{src_s}")
        names, gbs = train_on(Bs, f"{src_s.lower()}2{tgt_s.lower()}", ("softmax",))
        tgt = {f"set{tgt_s}": external_side(Bs, f"set{tgt_s}", PH[tgt_s], ONSET[tgt_s], CENS[tgt_s])}
        RESULTS[f"set{src_s}_to_set{tgt_s}"] = run_direction(f"set{src_s}", Bs, names, gbs, tgt, Bs["mask"])
        del Bs; gc.collect()

In [ ]:
# ============================ CELL 12 · SAVE (this is what to send back) ============================
def slim(o):
    if isinstance(o, dict): return {k: slim(v) for k, v in o.items() if k not in ("calibrator", "by_icu_hour_raw")}
    return o
SUMMARY = dict(time=time.strftime("%Y-%m-%dT%H:%M:%S"), core_commit=CORE_COMMIT, smoke=SMOKE_TEST, seed=SEED, features=FEATS, unit_fixes=UNIT_FIXES,
               physionet=PSTAT, mimic_matched=m_info, results=slim(RESULTS),
               training={nm: dict(auroc_internal=r["metrics"]["auroc_mean"], epochs_run=r["epochs_run"]) for nm, r in REC.items()},
               caveats=["setA is the same hospital as MIMIC (BIDMC): temporal / same-site check, possible patient overlap",
                        "Sepsis-3 labels were derived by different pipelines in the two cohorts: the external drop mixes label and covariate shift",
                        "attention replication on setB is deferred to Phase 3 (needs the correlational battery)"])
runs.save_json(SUMMARY, "01b_summary.json")

## What to send back

The printed output of Cells 5, 6, 7, 8, 9, 10 and 11, or `reports/01b_summary.json` plus `reports/01b_harmonisation.json`. All aggregate.